In [ ]:
#Importing required libraries
import pandas as pd
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

#Downloading data we working on
!wget -O moviedataset.zip https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%205/data/moviedataset.zip
print('unziping ...')
!unzip -o -j moviedataset.zip

#Pre-processing the data
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

# Extract year from title
movies_df['year'] = movies_df.title.str.extract(r'(\(\d\d\d\d\))', expand=False)
movies_df['year'] = movies_df.year.str.extract(r'(\d\d\d\d)', expand=False)
movies_df['title'] = movies_df.title.str.replace(r'(\(\d\d\d\d\))', '', regex=True)
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

# Split genres into a list
movies_df['genres'] = movies_df.genres.str.split('|')

# One-Hot Encoding for genres
moviesWithGenres_df = movies_df.copy()
for index, row in movies_df.iterrows():
    for genre in row['genres']:
        moviesWithGenres_df.at[index, genre] = 1
moviesWithGenres_df = moviesWithGenres_df.fillna(0)

# Clean ratings dataframe
ratings_df = ratings_df.drop('timestamp', axis=1)

# Content-Based Recommendation System

# Create input user profile
userInput = [
    {'title': 'Breakfast Club, The', 'rating': 5},
    {'title': 'Toy Story', 'rating': 3.5},
    {'title': 'Jumanji', 'rating': 2},
    {'title': 'Pulp Fiction', 'rating': 5},
    {'title': 'Akira', 'rating': 4.5}
]
inputMovies = pd.DataFrame(userInput)

# Add movieId to input user
# Create temporary lowercase columns for robust matching
movies_df['temp_title_lower'] = movies_df['title'].str.lower()
inputMovies['temp_title_lower'] = inputMovies['title'].str.lower()

inputId = movies_df[movies_df['temp_title_lower'].isin(inputMovies['temp_title_lower'].tolist())]

# Merge using the temporary lowercase titles
inputMovies = pd.merge(inputId, inputMovies, left_on='temp_title_lower', right_on='temp_title_lower')

# Drop the temporary lowercase columns after merging
movies_df = movies_df.drop('temp_title_lower', axis=1)
inputMovies = inputMovies.drop('temp_title_lower', axis=1)

inputMovies = inputMovies.drop(['genres', 'year'], axis=1)

# Get the genre table for user's movies
userMovies = moviesWithGenres_df[moviesWithGenres_df['movieId'].isin(inputMovies['movieId'].tolist())]
userMovies = userMovies.reset_index(drop=True)
userGenreTable = userMovies.drop(['movieId', 'title', 'genres', 'year'], axis=1)

# Calculate User Profile (Dot product)
userProfile = userGenreTable.transpose().dot(inputMovies['rating'])

# Prepare the full genre table
genreTable = moviesWithGenres_df.set_index(moviesWithGenres_df['movieId'])
genreTable = genreTable.drop(['movieId', 'title', 'genres', 'year'], axis=1)

#  Generate Recommendations
# Take weighted average
recommendationTable_df = ((genreTable * userProfile).sum(axis=1)) / (userProfile.sum())
recommendationTable_df = recommendationTable_df.sort_values(ascending=False)

# Final Recommendation Result (Top 20)
movies_df.loc[movies_df['movieId'].isin(recommendationTable_df.head(20).keys())]

--2026-04-27 05:28:16--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%205/data/moviedataset.zip
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 160301210 (153M) [application/zip]
Saving to: ‘moviedataset.zip’

moviedataset.zip    100%[===================>] 152.88M  58.8MB/s    in 2.6s    

2026-04-27 05:28:19 (58.8 MB/s) - ‘moviedataset.zip’ saved [160301210/160301210]

unziping ...
Archive:  moviedataset.zip
  inflating: links.csv               
  inflating: movies.csv              
  inflating: ratings.csv             
  inflating: README.txt              
  inflating: tags.csv        

,movieId,title,genres,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II (1995),[Comedy],1995
5,6,Heat (1995),"[Action, Crime, Thriller]",1995
6,7,Sabrina (1995),"[Comedy, Romance]",1995
7,8,Tom and Huck (1995),"[Adventure, Children]",1995
8,9,Sudden Death (1995),[Action],1995
9,10,GoldenEye (1995),"[Action, Adventure, Thriller]",1995
